In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import pandas as pd
import gc
from src.features.amount_features import create_transaction_amount_features
from src.features.time_features import create_time_features
from src.features.card_features import create_card_features
from src.features.email_features import create_email_features
from src.features.device_features import create_device_features
from src.features.address_features import create_address_features
from src.features.v_features import create_v_features
from src.features.c_d_features import create_c_d_features
from src.features.m_features import create_m_features
from src.features.id_features import create_id_features
from src.features.aggregations import create_aggregation_features
from src.utils import reduce_mem_usage

In [2]:
train_transaction = pd.read_csv("../data/raw/train_transaction.csv")
train_identity = pd.read_csv("../data/raw/train_identity.csv")
test_transaction = pd.read_csv("../data/raw/test_transaction.csv")
test_identity = pd.read_csv("../data/raw/test_identity.csv")

# Merge transaction and identity data on TransactionID
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

# Standardize the test set column names to match the train set
test.columns = test.columns.str.replace('-', '_')

print(f"Train Shape after merge: {train.shape}")
print(f"Test Shape after merge: {test.shape}")

# Free memory
del train_transaction, test_transaction, train_identity, test_identity
gc.collect()

Train Shape after merge: (590540, 434)
Test Shape after merge: (506691, 433)


32

In [3]:
train = create_transaction_amount_features(train)
test = create_transaction_amount_features(test)

# Verify the columns are actually there
print(train[['TransactionAmt', 'TransactionAmt_Log', 'TransactionAmt_is_round']].head())

Creating transaction amount features...
Creating transaction amount features...
   TransactionAmt  TransactionAmt_Log  TransactionAmt_is_round
0            68.5            4.241327                    False
1            29.0            3.401197                     True
2            59.0            4.094345                     True
3            50.0            3.931826                     True
4            50.0            3.931826                     True


In [4]:
# Create Time Features
train = create_time_features(train)
test = create_time_features(test)

Creating time features...
Creating time features...


In [5]:
# Create Card Features
train = create_card_features(train)
test = create_card_features(test)

Creating card interaction features...
Creating card interaction features...


In [6]:
# Apply
train = create_email_features(train)
test = create_email_features(test)

Creating email features...
Creating email features...


In [7]:
# Apply
train = create_device_features(train)
test = create_device_features(test)

Creating device features...
Creating device features...


In [8]:
# Apply
train = create_address_features(train)
test = create_address_features(test)

Creating address features...
Creating address features...


In [9]:
# Apply
train = create_v_features(train)
test = create_v_features(test)

Creating V-column aggregation features...
Creating V-column aggregation features...


In [10]:
# Apply
train = create_c_d_features(train)
test = create_c_d_features(test)

Creating C and D column features...


c:\Users\Anmoy\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Anmoy\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Creating C and D column features...


In [11]:
# Apply
train = create_m_features(train)
test = create_m_features(test)

Creating M column features...
Creating M column features...


In [12]:
# Apply
train = create_id_features(train)
test = create_id_features(test)

Creating ID features...
Creating ID features...


In [13]:
# 2. Dependent Features (Aggregations learned ONLY from train)
train_engineered, test_engineered = create_aggregation_features(train, test)

Creating aggregation features...


In [14]:
print(f"\nFinal Train Shape: {train_engineered.shape}")
print(f"Final Test Shape: {test_engineered.shape}")

# Count new features
original_cols = 434  # Approximate original column count after merge
new_features = train_engineered.shape[1] - original_cols
print(f"\nNew features created: ~{new_features}")

# Reduce memory
print("\nReducing memory usage...")
train_engineered = reduce_mem_usage(train_engineered)
test_engineered = reduce_mem_usage(test_engineered)

gc.collect()


Final Train Shape: (590540, 637)
Final Test Shape: (506691, 636)

New features created: ~203

Reducing memory usage...
Memory usage: 2597.96 MB -> 1433.86 MB (44.8% reduction)
Memory usage: 2225.22 MB -> 1240.42 MB (44.3% reduction)


28

In [15]:
train_engineered.to_csv("../data/processed/train_engineered.csv",index=False)
test_engineered.to_csv("../data/processed/test_engineered.csv",index=False)